9. In this exercise, we will predict the number of applications received
using the other variables in the College data set.

(a) Split the data set into a training set and a test set.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

college = pd.read_csv("College.csv", index_col = 0)

y = college["Apps"]
X = college.drop(columns=["Apps"])

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

print(X_train.shape, X_test.shape)


(543, 17) (234, 17)


(b) Fit a linear model using least squares on the training set, and
report the test error obtained.

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

linreg = LinearRegression()
linreg.fit(X_train, y_train)

y_pred = linreg.predict(X_test)

mse_test = mean_squared_error(y_test, y_pred)

print("Test MSE:", mse_test)


Test MSE: 642753.8976533666


(c) Fit a ridge regression model on the training set, with λ chosen
by cross-validation. Report the test error obtained.

In [4]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

alphas = np.logspace(-2, 4, 100)

ridge_model = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=alphas, cv=10)
)

ridge_model.fit(X_train, y_train)

ridge_cv = ridge_model.named_steps["ridgecv"]

best_alpha_ridge = ridge_cv.alpha_
print("Best lambda (alpha) by CV (Ridge):", best_alpha_ridge)

y_pred_ridge = ridge_model.predict(X_test)

mse_ridge = mean_squared_error(y_test, y_pred_ridge)
print("Ridge Test MSE:", mse_ridge)


Best lambda (alpha) by CV (Ridge): 10.722672220103231
Ridge Test MSE: 679848.1213942295


(d) Fit a lasso model on the training set, with λ chosen by cross-
validation. Report the test error obtained, along with the num-
ber of non-zero coefficient estimates.

In [5]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

alphas = np.logspace(-4, 2, 100)

lasso_model = make_pipeline(
    StandardScaler(),
    LassoCV(alphas=alphas, cv=10, random_state=1, max_iter=100000)
)

lasso_model.fit(X_train, y_train)

lasso_cv = lasso_model.named_steps["lassocv"]

best_alpha_lasso = lasso_cv.alpha_
print("Best lambda (alpha) by CV (Lasso):", best_alpha_lasso)

y_pred_lasso = lasso_model.predict(X_test)

mse_lasso = mean_squared_error(y_test, y_pred_lasso)
print("Lasso Test MSE:", mse_lasso)

non_zero_coefs = np.sum(lasso_cv.coef_ != 0)
print("Number of non-zero coefficients (Lasso):", non_zero_coefs)


Best lambda (alpha) by CV (Lasso): 12.32846739442066
Lasso Test MSE: 659693.8740382595
Number of non-zero coefficients (Lasso): 14


(e) Fit a PCR model on the training set, with M chosen by cross-
validation. Report the test error obtained, along with the value
of M selected by cross-validation.

In [6]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

pcr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("linreg", LinearRegression())
])

n_features = X_train.shape[1]
param_grid = {
    "pca__n_components": list(range(1, n_features + 1))
}

pcr_cv = GridSearchCV(
    pcr_pipe,
    param_grid=param_grid,
    cv=10,
    scoring="neg_mean_squared_error"
)

pcr_cv.fit(X_train, y_train)

best_M = pcr_cv.best_params_["pca__n_components"]
print("Best number of components M (PCR):", best_M)

best_pcr_model = pcr_cv.best_estimator_
y_pred_pcr = best_pcr_model.predict(X_test)

mse_pcr = mean_squared_error(y_test, y_pred_pcr)
print("PCR Test MSE:", mse_pcr)

Best number of components M (PCR): 17
PCR Test MSE: 642753.8976533791
